In [2]:
from faker import Faker
import pandas as pd
import numpy as np
from random import randint
from datetime import datetime, timedelta
from pathlib import Path
import sys

# Setup
fake = Faker()
np.random.seed(42)
Faker.seed(42)


# Adiciona caminho raiz para importações
BASE_DIR = Path.cwd().resolve().parents[1] #subiu para a raiz

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
raw_dir = BASE_DIR / 'Data' / 'RAW' / 'FULL_SIMULATED'
raw_dir.mkdir(parents=True, exist_ok=True)

# Parâmetros
n_leads = 350_000
start_date = datetime(2023, 4, 1)
end_date = datetime.today()

origens = ['Facebook', 'Google Ads', 'Indicação', 'Instagram', 'Orgânico', 'TikTok', 'WhatsApp', 'YouTube Ads', 'E-mail Marketing']
pesos_origens = [0.25, 0.20, 0.05, 0.15, 0.10, 0.08, 0.07, 0.06, 0.04]

perfis = ['Conservador', 'Moderado', 'Agressivo']
pesos_perfil = [0.15, 0.6, 0.25]

# Conversão por canal
conversao_por_canal = {
    'Facebook': 0.07,
    'Google Ads': 0.12,
    'Indicação': 0.20,
    'Instagram': 0.09,
    'Orgânico': 0.10,
    'TikTok': 0.08,
    'WhatsApp': 0.18,
    'YouTube Ads': 0.11,
    'E-mail Marketing': 0.05
}

# Valor de depósito por perfil
valores_deposito = {
    'Conservador': (100, 800),
    'Moderado': (200, 2000),
    'Agressivo': (500, 5000)
}

# Gerar países
paises = list({fake.country() for _ in range(200)})

def get_week_of_month(dt):
    return (dt.day - 1) // 7 + 1

# Geração de dados
dados = []
for i in range(n_leads):
    data_cadastro = fake.date_between(start_date=start_date, end_date=end_date)
    origem = np.random.choice(origens, p=pesos_origens)
    perfil = np.random.choice(perfis, p=pesos_perfil)
    pais = np.random.choice(paises)

    taxa_conv = conversao_por_canal[origem]
    convertido = np.random.rand() < taxa_conv

    dias_ate_trade = randint(0, 30) if convertido else np.nan
    min_dep, max_dep = valores_deposito[perfil]
    valor_deposito = round(np.random.uniform(min_dep, max_dep), 2) if convertido else 0.0

    dados.append([
        i + 1,
        data_cadastro,
        origem,
        perfil,
        pais,
        dias_ate_trade,
        valor_deposito,
        'Convertido' if convertido else 'Não Convertido'
    ])

# Criando DataFrame
colunas = ['lead_id', 'data_cadastro', 'origem', 'perfil', 'pais', 'dias_ate_1o_trade', 'valor_deposito', 'status_conversao']
df = pd.DataFrame(dados, columns=colunas)

# Features adicionais
df['foi_convertido'] = df['status_conversao'] == 'Convertido'
df['ano'] = pd.to_datetime(df['data_cadastro']).dt.year
df['mes'] = pd.to_datetime(df['data_cadastro']).dt.month.astype(str).str.zfill(2)
df['semana'] = pd.to_datetime(df['data_cadastro']).apply(get_week_of_month)
df['semana_str'] = df['semana'].astype(str).str.zfill(2)
df['target'] = df['foi_convertido'].astype(int)

# Salvando arquivos em pastas separadas
for (ano, mes, semana_str), grupo in df.groupby(['ano', 'mes', 'semana_str']):
    output_dir = raw_dir / str(ano) / mes
    output_dir.mkdir(parents=True, exist_ok=True)

    nome_arquivo = f"{ano}_{mes}_{semana_str}.csv"
    grupo.to_csv(output_dir / nome_arquivo, index=False)
    print(f"✅ {nome_arquivo} salvo com {len(grupo)} leads")


✅ 2023_04_01.csv salvo com 3298 leads
✅ 2023_04_02.csv salvo com 3282 leads
✅ 2023_04_03.csv salvo com 3230 leads
✅ 2023_04_04.csv salvo com 3277 leads
✅ 2023_04_05.csv salvo com 899 leads
✅ 2023_05_01.csv salvo com 3210 leads
✅ 2023_05_02.csv salvo com 3300 leads
✅ 2023_05_03.csv salvo com 3170 leads
✅ 2023_05_04.csv salvo com 3301 leads
✅ 2023_05_05.csv salvo com 1365 leads
✅ 2023_06_01.csv salvo com 3367 leads
✅ 2023_06_02.csv salvo com 3206 leads
✅ 2023_06_03.csv salvo com 3396 leads
✅ 2023_06_04.csv salvo com 3311 leads
✅ 2023_06_05.csv salvo com 890 leads
✅ 2023_07_01.csv salvo com 3250 leads
✅ 2023_07_02.csv salvo com 3215 leads
✅ 2023_07_03.csv salvo com 3373 leads
✅ 2023_07_04.csv salvo com 3285 leads
✅ 2023_07_05.csv salvo com 1461 leads
✅ 2023_08_01.csv salvo com 3199 leads
✅ 2023_08_02.csv salvo com 3263 leads
✅ 2023_08_03.csv salvo com 3262 leads
✅ 2023_08_04.csv salvo com 3299 leads
✅ 2023_08_05.csv salvo com 1448 leads
✅ 2023_09_01.csv salvo com 3192 leads
✅ 2023_09_02.c